# 初始 CSV 清理

串流清理六份 Reddit CSV，輸出至新的 `Data/process/01_initial_csv_clean/<UTC時間>/`。

- 保留指定欄位；未命名第一欄改名為 `row_num`；輸出 UTF-8 BOM。
- 必要欄位 `author`、`body`、`created_utc`、`id`、`link_id` 若為空字串或純空白，刪除該留言。
- `created_utc` 須為有限且可轉換的 Unix timestamp；無效時間戳刪除，小數秒合法。依 UTC 月份刪除 2012-04 與 2020-08 整月留言，其餘月份沿用原規則。
- `link_id` 去除前後空白與 t3_ 前綴後不可為空，例如 `t3_` 也刪除。
- 作者為 deleted/removed 或 delete/remove（含方括號形式、不分大小寫、忽略前後空白）時刪除；採完整值比對。
- 非必要欄位如 `author_fullname`、`parent_id`、`nest_level`、`row_num` 可缺值。保留合法列的原始值與重複留言，不改寫輸出文字。
- 缺值或無效必要欄位的列寫入 `.dropped.jsonl`，含來源邏輯列、CSV 結束行號、原因與原欄位；摘要記錄各原因筆數（同一列可以有多個原因）。
- 欄位數錯誤另存 `.rejected.jsonl`；CSV 語法錯誤直接停止。
- 每次產生新 run，不改寫既有輸出。成功後更新 cleaned pointer 並使 matching/thread pointer 失效。


In [1]:
import json
from pathlib import Path
import csv
import sys
import time
import math
from collections import Counter
from datetime import datetime, timezone

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Data" / "raw").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Cannot find Data/raw or input CSV files")

import sys
sys.path.insert(0,str(ROOT/'Notebook'))
from pipeline_io import new_run, publish, input_run, remove_work_database

RAW_DIR = ROOT / "Data" / "raw"
OUTPUT_DIR = new_run(ROOT,'cleaned')
COLUMNS = ["row_num", "author", "author_fullname", "body", "created_utc",
           "id", "link_id", "nest_level", "parent_id"]
REQUIRED_COLUMNS = ['author', 'body', 'created_utc', 'id', 'link_id']
EXCLUDED_UTC_MONTHS = frozenset({'2012-04', '2020-08'})
PROGRESS_EVERY = 250_000

field_limit = sys.maxsize
while True:
    try:
        csv.field_size_limit(field_limit)
        break
    except OverflowError:
        field_limit //= 10

sources = [RAW_DIR/f'wallstreetbets_comment_part{i}.csv' for i in range(1,7)]
assert all(p.is_file() for p in sources), 'Expected all six Reddit CSV files'
if not sources:
    raise FileNotFoundError("Cannot find Data/raw or input CSV files")
print(f"Found {len(sources)} files, {sum(p.stat().st_size for p in sources) / 1e9:.2f} GB")


Found 6 files, 2.95 GB


In [2]:
def drop_reasons(values):
    reasons = ['missing_' + name for name in REQUIRED_COLUMNS if not values[name].strip()]
    author = values['author'].strip().lower()
    if author in {'deleted', '[deleted]', 'delete', '[delete]'}:
        reasons.append('deleted_author')
    elif author in {'removed', '[removed]', 'remove', '[remove]'}:
        reasons.append('removed_author')
    raw_timestamp = values['created_utc'].strip()
    if raw_timestamp:
        try:
            timestamp = float(raw_timestamp)
            if not math.isfinite(timestamp): raise ValueError('Nonfinite timestamp')
            comment_datetime = datetime.fromtimestamp(timestamp, tz=timezone.utc)
            month = comment_datetime.strftime('%Y-%m')
            if month in EXCLUDED_UTC_MONTHS:
                reasons.append('excluded_utc_month_' + month)
        except (ValueError, OverflowError, OSError):
            reasons.append('invalid_created_utc')
    raw_thread = values['link_id'].strip().lower()
    if raw_thread and not (raw_thread[3:] if raw_thread.startswith('t3_') else raw_thread):
        reasons.append('empty_normalized_thread_id')
    return reasons


def clean_csv(source, output_dir=OUTPUT_DIR, progress_every=PROGRESS_EVERY):
    source, output_dir = Path(source), Path(output_dir)
    destination = output_dir / source.name
    if destination.resolve() == source.resolve(): raise ValueError('Output must differ from source')
    output_dir.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    rejected_path, dropped_path = destination.with_suffix('.rejected.jsonl'), destination.with_suffix('.dropped.jsonl')
    started = time.monotonic()
    source_stat = (source.stat().st_size, source.stat().st_mtime_ns)
    count = rejected = dropped = input_records = 0
    reason_counts = Counter()
    try:
        with source.open('r', encoding='utf-8-sig', newline='') as incoming:
            reader = csv.reader(incoming, strict=True)
            header = next(reader, None)
            if not header: raise ValueError('Missing CSV header')
            if header[0] in ('', 'Unnamed: 0', '編號', 'row_num'): header[0] = 'row_num'
            missing = [name for name in COLUMNS if name not in header]
            if missing: raise ValueError(f'Missing columns: {missing}')
            if any(header.count(name) != 1 for name in COLUMNS): raise ValueError('Duplicate selected columns')
            if not set(REQUIRED_COLUMNS) <= set(COLUMNS): raise ValueError('Required columns must be selected')
            positions = [header.index(name) for name in COLUMNS]
            with temporary.open('w', encoding='utf-8-sig', newline='') as outgoing,                  rejected_path.open('w', encoding='utf-8') as rejects,                  dropped_path.open('w', encoding='utf-8') as drops:
                writer = csv.writer(outgoing); writer.writerow(COLUMNS)
                for row in reader:
                    input_records += 1
                    if len(row) != len(header):
                        rejected += 1
                        rejects.write(json.dumps({'source_row': input_records, 'physical_line_end': reader.line_num,
                                                  'fields': row}, ensure_ascii=False) + '\n')
                        continue
                    selected = [row[i] for i in positions]
                    reasons = drop_reasons(dict(zip(COLUMNS, selected)))
                    if reasons:
                        dropped += 1; reason_counts.update(reasons)
                        drops.write(json.dumps({'source_row': input_records, 'physical_line_end': reader.line_num,
                                                'reasons': reasons, 'fields': row}, ensure_ascii=False) + '\n')
                    else:
                        writer.writerow(selected); count += 1
                    if progress_every and input_records % progress_every == 0:
                        print(f'{source.name}: input={input_records:,}, kept={count:,}, dropped={dropped:,}, rejected={rejected:,}', flush=True)
        if source_stat != (source.stat().st_size, source.stat().st_mtime_ns): raise ValueError('Source changed during cleaning')
        assert input_records == count + rejected + dropped
        temporary.replace(destination)
        if rejected == 0: rejected_path.unlink(missing_ok=True)
        if dropped == 0: dropped_path.unlink(missing_ok=True)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    return {'file': source.name, 'input_records': input_records, 'rows': count,
            'rejected_records': rejected, 'dropped_records': dropped, 'drop_reason_counts': dict(reason_counts),
            'output_bytes': destination.stat().st_size, 'seconds': round(time.monotonic() - started, 1)}

# Empty/whitespace values, invalid timestamps, normalized empty thread and optional missing values.
valid_example = {name: '' for name in COLUMNS}
valid_example.update(author='valid_user', body='TEST', created_utc='1609459200.5', id='abc', link_id='t3_thread')
assert drop_reasons(valid_example) == []
for name in REQUIRED_COLUMNS:
    assert 'missing_' + name in drop_reasons(dict(valid_example, **{name: '  '}))
for value in ['NaN', 'inf', '-inf', 'bad', '1e100']:
    assert 'invalid_created_utc' in drop_reasons(dict(valid_example, created_utc=value))
assert drop_reasons(dict(valid_example, link_id=' t3_ ')) == ['empty_normalized_thread_id']
print('Required-field cleaning checks passed')


Required-field cleaning checks passed


In [3]:
results = []
for source in sources:
    result = clean_csv(source)
    results.append(result)
    print(result, flush=True)

print(f"Total: {sum(r['rows'] for r in results):,} rows")
print(f"Output: {sum(r['output_bytes'] for r in results) / 1e9:.2f} GB")
print(f"Output directory: {OUTPUT_DIR}")

total_reasons = Counter()
for result in results: total_reasons.update(result['drop_reason_counts'])
publish(ROOT, 'cleaned', OUTPUT_DIR, {
    'schema_version': 2, 'files': results, 'total_rows': sum(r['rows'] for r in results),
    'total_input_records': sum(r['input_records'] for r in results),
    'total_dropped_records': sum(r['dropped_records'] for r in results),
    'total_rejected_records': sum(r['rejected_records'] for r in results),
    'drop_reason_counts': dict(total_reasons), 'required_columns': REQUIRED_COLUMNS,
    'missing_policy': 'Drop empty/whitespace required fields; optional missing values retained.',
    'author_policy': 'Require nonblank author; drop exact deleted/removed/delete/remove markers, bracketed or plain, case-insensitive after trimming.',
    'timestamp_policy': 'Finite convertible Unix timestamp; exclude specified UTC months only.',
    'excluded_utc_months': sorted(EXCLUDED_UTC_MONTHS),
    'month_exclusion_policy': 'Exclude entire specified UTC calendar months during raw CSV cleaning.',
    'excluded_month_counts': {month: total_reasons.get('excluded_utc_month_' + month, 0) for month in sorted(EXCLUDED_UTC_MONTHS)},
    'input_directory': str(RAW_DIR),
})


wallstreetbets_comment_part1.csv: input=250,000, kept=241,047, dropped=8,953, rejected=0
{'file': 'wallstreetbets_comment_part1.csv', 'input_records': 310753, 'rows': 299810, 'rejected_records': 0, 'dropped_records': 10943, 'drop_reason_counts': {'excluded_utc_month_2012-04': 217, 'deleted_author': 10385, 'missing_body': 336, 'missing_created_utc': 22, 'missing_id': 23, 'missing_link_id': 26, 'missing_author': 6, 'invalid_created_utc': 1}, 'output_bytes': 62042383, 'seconds': 2.9}
wallstreetbets_comment_part2.csv: input=250,000, kept=235,246, dropped=14,745, rejected=9
wallstreetbets_comment_part2.csv: input=500,000, kept=468,375, dropped=31,603, rejected=22
{'file': 'wallstreetbets_comment_part2.csv', 'input_records': 575148, 'rows': 537839, 'rejected_records': 24, 'dropped_records': 37285, 'drop_reason_counts': {'deleted_author': 36593, 'missing_body': 731}, 'output_bytes': 94936845, 'seconds': 6.6}
wallstreetbets_comment_part3.csv: input=250,000, kept=230,527, dropped=19,473, reject

## 預覽
輸出前三行


In [4]:
from itertools import islice

with (OUTPUT_DIR / sources[0].name).open(encoding="utf-8-sig", newline="") as f:
    preview = list(islice(csv.DictReader(f), 3))
preview


[{'row_num': '217',
  'author': 'dustybus',
  'author_fullname': 't2_4zask',
  'body': 'Thanks for the awesome reply',
  'created_utc': '1335840541.0',
  'id': 'c4il4l6',
  'link_id': 't3_t0olb',
  'nest_level': '2',
  'parent_id': 't1_c4ikmnk'},
 {'row_num': '218',
  'author': 'simplemethod',
  'author_fullname': 't2_4ali0',
  'body': 'I am personally bearish on Netflix I did like them and continue to use their DVD mail service but have been growing increasingly frustrated with them But that is not why Im bearish after reviewing their 10K and seeing how they performed post earnings I have a strong feeling the stock will fall below 80 \n\nWith a low barrier to entry in that business coupled with the current lack of innovation to acquire new subscribers I just dont have a positive outlook on that stock Also it did break below 80 briefly today which takes away support at that level and I would look for them to break below it again soon \n\nNot a whole lot that I can add that jartek hasnt

## 下游使用
01 不讀取 stock 資料，不依 ticker 或研究日期範圍過濾。必要欄位驗證成功的原始值維持不變。

重新執行 01 後，需重新執行 02；02 的 `source_row` 是新 cleaned CSV 的邏輯列號。原始資料位置可由保留的 `row_num` 與 01 稽核檔追溯。舊 matching 與後續產物不會被刪除或自動重建。
